# The finite volume method: 1D steady convection–diffusion

$$\frac{\mathrm{d}}{\mathrm{d}x}(\rho u \phi) = \frac{\mathrm{d}}{\mathrm{d}x}\left(\Gamma \frac{\mathrm{d}\phi}{\mathrm{d}x}\right), \qquad \phi(0) = 1,\ \phi(L) = 0$$

This is the classic example from Versteeg & Malalasekera, chapter 5: $L = 1$ m, $\rho = 1$, $\Gamma = 0.1$. The exact solution is

$$\phi(x) = 1 - \frac{e^{\rho u x/\Gamma} - 1}{e^{\rho u L/\Gamma} - 1}$$

Split the domain into $N$ cells. For each cell P with neighbours W and E: $a_P\phi_P = a_W\phi_W + a_E\phi_E + S_u$, where $F = \rho u$ is the convective flux and $D = \Gamma/\Delta x$ is the diffusive conductance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
L, rho, Gamma = 1.0, 1.0, 0.1
phi_A, phi_B = 1.0, 0.0

def exact(x, u):
    return 1 - (np.exp(rho * u * x / Gamma) - 1) / (np.exp(rho * u * L / Gamma) - 1)

**Exercise.** Build and solve the system for the central and the upwind scheme. Boundary cells use a half-cell distance to the wall, so their diffusive conductance is $2D$.

| scheme | $a_W$ | $a_E$ |
|---|---|---|
| central | $D + F/2$ | $D - F/2$ |
| upwind ($F>0$) | $D + F$ | $D$ |

Interior cells have $a_P = a_W + a_E$. In the first cell, the west link becomes a source: $S_P = -(2D + F)$ and $S_u = (2D + F)\phi_A$. At the east boundary the link to $\phi_B$ is $2D - F$ for central and $2D$ for upwind.

In [ ]:
def fvm(N, u, scheme):
    dx = L / N
    F, D = rho * u, Gamma / dx
    A = np.zeros((N, N)); b = np.zeros(N)
    # YOUR CODE HERE
    raise NotImplementedError
    xc = (np.arange(N) + 0.5) * dx
    return xc, np.linalg.solve(A, b)

xf = np.linspace(0, L, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, u in zip(axes, (0.1, 2.5)):
    ax.plot(xf, exact(xf, u), "k", lw=3, alpha=0.3, label="exact")
    for scheme in ("central", "upwind"):
        xc, phi = fvm(5, u, scheme)
        ax.plot(xc, phi, "o--", label=scheme)
    ax.set_title(f"u = {u} m/s, cell Peclet = {rho * u * 0.2 / Gamma:.1f}"); ax.legend()

What you should see: at $u = 0.1$ both schemes match the exact curve. At $u = 2.5$ the cell Péclet number is 5 and central differencing oscillates, while upwind stays bounded but smeared.

**Question.** How many cells does central need at $u = 2.5$ to stop oscillating? (Hint: keep the cell Péclet number below 2.)